<a href="https://colab.research.google.com/github/Krishnadev-cmd/Pytorch-Turorial/blob/main/Cats_Vs_Dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("yashdogra/cats-and-dogs")
print("Path to dataset files:", path)

100%|██████████| 756M/756M [00:08<00:00, 98.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yashdogra/cats-and-dogs/versions/1


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [3]:
dataset_path = "/root/.cache/kagglehub/datasets/yashdogra/cats-and-dogs/versions/1/images"


In [15]:
from torch.utils.data import Dataset
import os
from PIL import Image
import torch

class MyDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Filter out non-image files if necessary
        self.images = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_name = self.images[index]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        # Simple label logic: 1 for dog, 0 for cat based on filename
        label = 1 if "dog" in img_name.lower() else 0

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)

In [16]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

dataset = MyDataset(dataset_path, transform=transform)

In [17]:
model = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(16 * 32 * 32, 2)
    )

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [20]:
for epoch in range(1):
  model.train()
  for image , label in train_loader:
    output = model(image)
    loss = criterion(output , label)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()